# EcoHome Energy Advisor: Run and Evaluate

This notebook runs realistic Berlin energy questions end to end and checks both answer quality and tool choice.

In [1]:
import json
import os
from datetime import datetime, timedelta
from pathlib import Path

from agent import Agent
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field


In [2]:
ECOHOME_SYSTEM_PROMPT = '''You are EcoHome's Energy Advisor for Berlin homes. Your job is to help people lower energy costs and use more of their own solar power without making their home uncomfortable.

For each question:
1. Work out what information is needed.
2. Use weather data for future solar questions and electricity prices for scheduling or cost questions.
3. Use energy history or solar history when the user asks about their past pattern.
4. Search the energy tips knowledge base when practical advice would help.
5. Use the savings calculator whenever you state a saving that can be calculated.
6. For a question with an explicit past period, query the relevant history instead of guessing.
7. Use the personalized tomorrow plan for broad next day schedules and the carbon tool when the user asks about environmental impact.
8. Give a short recommendation first, then explain the timing, cost, solar, and any assumptions in plain language.

Key capabilities: EV charging, HVAC settings, appliance scheduling, solar use, energy storage, historical analysis, personalized planning, EUR savings, and carbon estimates.

Recommendation rules:
* Give specific Berlin local hours when the data supports them.
* Prefer strong solar hours for flexible daytime tasks when the forecast supports it, but never schedule an EV after its departure time.
* Compare those hours with off peak grid prices when solar is weak.
* Mention retrieved tips with their source filename in square brackets.
* Do not claim to control a device or guarantee a saving.
* For future scheduling, use weather plus prices. For past usage, use the database. For practical steps, retrieve a tip and cite its filename.

Example questions include EV charging tomorrow, thermostat settings during a price spike, dishwasher timing, pool pump timing, and battery scheduling.'''

ecohome_agent = Agent(instructions=ECOHOME_SYSTEM_PROMPT)
print('Available tools:', ecohome_agent.get_agent_tools())


Available tools: ['get_weather_forecast', 'get_electricity_prices', 'query_energy_usage', 'query_solar_generation', 'get_recent_energy_summary', 'get_user_preferences', 'get_personalized_tomorrow_plan', 'search_energy_tips', 'calculate_energy_savings', 'calculate_carbon_impact']


In [3]:
today = datetime.now().date()
history_start = (today - timedelta(days=29)).isoformat()
history_end = today.isoformat()

test_cases = [
    {'id': 'ev_charging', 'question': 'When should I charge my electric car tomorrow to minimize cost and maximize solar power?', 'expected_tools': ['get_personalized_tomorrow_plan', 'get_weather_forecast', 'get_electricity_prices'], 'evaluation_criteria': ['Give a practical charging window before the saved departure time.', 'Explain the solar and price trade off.', 'State assumptions or uncertainty when live data is limited.']},
    {'id': 'thermostat_peak', 'question': 'What thermostat approach should I use tomorrow afternoon if electricity prices are high?', 'expected_tools': ['get_weather_forecast', 'get_electricity_prices', 'search_energy_tips'], 'evaluation_criteria': ['Recommend a comfort safe thermostat approach.', 'Connect the recommendation to the high price period and forecast.', 'Give an actionable temperature or timing suggestion.']},
    {'id': 'dishwasher', 'question': 'How much can I save by running my dishwasher during off peak hours?', 'expected_tools': ['get_electricity_prices', 'query_energy_usage', 'calculate_energy_savings', 'search_energy_tips'], 'evaluation_criteria': ['Use the household dishwasher history and tariff information.', 'Give a EUR saving estimate and a clear lower cost window.', 'Avoid claiming a guaranteed saving.']},
    {'id': 'washing_machine', 'question': 'When should I run a washing machine tomorrow in Berlin?', 'expected_tools': ['get_personalized_tomorrow_plan', 'search_energy_tips'], 'evaluation_criteria': ['Recommend a specific suitable window tomorrow.', 'Explain whether the choice benefits from solar or lower prices.', 'Include a practical safety or appliance use tip.']},
    {'id': 'pool_pump', 'question': 'What is the best time to run my pool pump this week based on the forecast?', 'expected_tools': ['get_weather_forecast', 'get_electricity_prices', 'search_energy_tips'], 'evaluation_criteria': ['Use forecast and pricing evidence.', 'Recommend a recurring daytime or low cost operating window.', 'Explain the solar or tariff rationale in plain language.']},
    {'id': 'solar_maximization', 'question': 'How can I use more of my solar power tomorrow?', 'expected_tools': ['get_personalized_tomorrow_plan', 'get_weather_forecast', 'get_electricity_prices', 'search_energy_tips'], 'evaluation_criteria': ['Offer multiple practical load shifting actions.', "Anchor the advice to tomorrow's forecast window.", 'Balance solar self consumption with household constraints.']},
    {'id': 'history_reduction', 'question': 'Suggest three ways I can reduce energy use based on my usage history.', 'expected_tools': ['get_recent_energy_summary', 'search_energy_tips'], 'evaluation_criteria': ['Give exactly or clearly at least three distinct suggestions.', 'Tie suggestions to observed household patterns.', 'Include practical advice supported by a source.']},
    {'id': 'hvac_history', 'question': f'From {history_start} through {history_end}, what does my HVAC usage history suggest about reducing evening costs?', 'expected_tools': ['query_energy_usage', 'search_energy_tips', 'calculate_energy_savings'], 'evaluation_criteria': ['Use the requested historical period.', 'Identify an evening cost reduction action.', 'Provide a cautious quantified cost or saving estimate when supported.']},
    {'id': 'battery', 'question': 'How should I charge and use a home battery over the next two days?', 'expected_tools': ['get_weather_forecast', 'get_electricity_prices', 'get_user_preferences', 'search_energy_tips'], 'evaluation_criteria': ['Use forecast, tariff, and saved battery preference evidence.', 'Explain when to charge from solar and when to reserve for expensive periods.', 'Respect the configured battery reserve.']},
    {'id': 'solar_history', 'question': f'From {history_start} through {history_end}, how does my solar generation compare with my consumption and what should I move?', 'expected_tools': ['query_energy_usage', 'query_solar_generation', 'search_energy_tips'], 'evaluation_criteria': ['Compare the requested generation and consumption history.', 'Identify a load to move toward solar hours.', 'Give a concise evidence based recommendation.']},
    {'id': 'personalized_plan', 'question': 'Give me a tomorrow plan that respects my saved EV departure time, comfort range, and battery reserve.', 'expected_tools': ['get_personalized_tomorrow_plan'], 'evaluation_criteria': ['Reflect the saved departure time, comfort range, and battery reserve.', 'Give an understandable device schedule.', 'Explain why the plan uses solar or lower cost hours.']},
    {'id': 'carbon_impact', 'question': 'What is the estimated CO2e impact of shifting 4 kWh of flexible use to 3 kWh of solar energy?', 'expected_tools': ['calculate_carbon_impact'], 'evaluation_criteria': ['Report a CO2e estimate using the stated energy quantities.', 'Explain the grid avoidance assumption.', 'Present the result as an estimate, not a guarantee.']},
]
assert len(test_cases) >= 10


In [4]:
def get_tool_names(messages):
    names = []
    for message in messages:
        for call in getattr(message, 'tool_calls', []) or []:
            names.append(call.get('name', 'unknown'))
        name = getattr(message, 'name', None)
        if name and message.__class__.__name__ == 'ToolMessage':
            names.append(name)
    return list(dict.fromkeys(names))

def get_tool_trace(messages):
    trace = []
    for message in messages:
        if message.__class__.__name__ != 'ToolMessage':
            continue
        content = getattr(message, 'content', '')
        if not isinstance(content, str):
            content = json.dumps(content)
        trace.append({'tool': getattr(message, 'name', 'unknown'), 'output_preview': content[:1200]})
    return trace

def final_text(messages):
    for message in reversed(messages):
        content = getattr(message, 'content', '')
        if isinstance(content, list):
            content = ''.join(
                item.get('text', '') for item in content
                if isinstance(item, dict) and item.get('type') in {'text', 'output_text'}
            )
        if message.__class__.__name__ == 'AIMessage' and isinstance(content, str) and content.strip():
            return content
    return ''

class ResponseJudgment(BaseModel):
    accuracy: float = Field(ge=0, le=100)
    relevance: float = Field(ge=0, le=100)
    completeness: float = Field(ge=0, le=100)
    usefulness: float = Field(ge=0, le=100)
    feedback: list[str] = Field(min_length=1, max_length=4)

class ToolUsageJudgment(BaseModel):
    tool_appropriateness: float = Field(ge=0, le=100)
    tool_completeness: float = Field(ge=0, le=100)
    feedback: list[str] = Field(min_length=1, max_length=4)

class ReportJudgment(BaseModel):
    strengths: list[str] = Field(min_length=1, max_length=5)
    weaknesses: list[str] = Field(min_length=1, max_length=6)
    recommendations: list[str] = Field(min_length=1, max_length=5)

TOOL_CAPABILITIES = {
    'get_weather_forecast': 'live hourly weather and solar irradiance forecast',
    'get_electricity_prices': 'date aware Berlin hourly electricity tariff',
    'query_energy_usage': 'historical device level electricity use and cost',
    'query_solar_generation': 'historical rooftop solar generation',
    'get_recent_energy_summary': 'recent household usage summary',
    'get_user_preferences': 'saved EV, comfort, battery, and priority preferences',
    'get_personalized_tomorrow_plan': 'combined weather, pricing, and preference schedule',
    'search_energy_tips': 'sourced energy saving guidance',
    'calculate_energy_savings': 'EUR impact of reducing or shifting energy use',
    'calculate_carbon_impact': 'estimated CO2e impact of solar load shifting',
}

def evaluation_llm():
    api_key = os.getenv('VOCAREUM_API_KEY') or os.getenv('OPENAI_API_KEY')
    if not api_key:
        raise RuntimeError('Set VOCAREUM_API_KEY or OPENAI_API_KEY before running LLM evaluation.')
    return ChatOpenAI(
        model=os.getenv('OPENAI_MODEL', 'gpt-5.6-luna'),
        temperature=0,
        api_key=api_key,
        base_url=os.getenv('OPENAI_BASE_URL', 'https://openai.vocareum.com/v1'),
        reasoning_effort=os.getenv('OPENAI_EVALUATION_REASONING_EFFORT', 'none'),
        use_responses_api=True,
        timeout=60,
        max_retries=1,
    )

def run_judge(schema, prompt):
    return evaluation_llm().with_structured_output(schema).invoke(prompt).model_dump()

def evaluate_response(question, final_response, expected_response):
    if not final_response.strip():
        return {'accuracy': 0.0, 'relevance': 0.0, 'completeness': 0.0, 'usefulness': 0.0, 'feedback': ['No final response was returned.'], 'evaluation_method': 'input_validation'}
    prompt = f'''You are an independent energy advisor evaluator. Judge the answer semantically, not by keyword overlap.

Customer question: {question}
Evaluation criteria: {json.dumps(expected_response)}
Advisor answer: {final_response}

Score accuracy, relevance, completeness, and usefulness from 0 to 100. Check factual caution, whether the answer directly solves the question, coverage of the stated criteria, and whether a Berlin household could act on it. Give specific feedback about missing evidence, timing, costs, or assumptions.'''
    try:
        judgment = run_judge(ResponseJudgment, prompt)
        judgment['evaluation_method'] = 'llm_as_judge'
        return judgment
    except Exception as exc:
        return {'accuracy': 0.0, 'relevance': 0.0, 'completeness': 0.0, 'usefulness': 0.0, 'feedback': [f'LLM evaluation unavailable: {exc}'], 'evaluation_method': 'unavailable'}

def evaluate_tool_usage(messages_list, expected_tools, question=''):
    actual_tools = get_tool_names(messages_list)
    trace = get_tool_trace(messages_list)
    prompt = f'''You are evaluating an AI energy advisor's tool choices. Judge the tool use semantically, not with set intersection.

Question: {question}
Expected tools or acceptable evidence routes: {expected_tools}
Actual tools: {actual_tools}
Tool trace: {json.dumps(trace)}
Tool capability guide: {json.dumps(TOOL_CAPABILITIES)}

Score appropriateness and completeness from 0 to 100. Accept a semantically equivalent combined tool when it supplies the required evidence, for example a personalized tomorrow plan instead of separate forecast and tariff calls. Do not penalize relevant supporting tools. Explain any genuinely missing evidence.'''
    try:
        judgment = run_judge(ToolUsageJudgment, prompt)
        judgment.update({'actual_tools': actual_tools, 'evaluation_method': 'llm_as_judge'})
        return judgment
    except Exception as exc:
        return {'actual_tools': actual_tools, 'tool_appropriateness': 0.0, 'tool_completeness': 0.0, 'feedback': [f'LLM tool evaluation unavailable: {exc}'], 'evaluation_method': 'unavailable'}

def generate_evaluation_report(results):
    response_metrics = ['accuracy', 'relevance', 'completeness', 'usefulness']
    averages = {metric: round(sum(result['response_evaluation'][metric] for result in results) / len(results), 1) for metric in response_metrics}
    averages['tool_appropriateness'] = round(sum(result['tool_evaluation']['tool_appropriateness'] for result in results) / len(results), 1)
    averages['tool_completeness'] = round(sum(result['tool_evaluation']['tool_completeness'] for result in results) / len(results), 1)
    response_score = round(sum(averages[metric] for metric in response_metrics) / len(response_metrics), 1)
    tool_score = round((averages['tool_appropriateness'] + averages['tool_completeness']) / 2, 1)
    overall_score = round((response_score * 0.65) + (tool_score * 0.35), 1)
    weak_cases = [result['test_id'] for result in results if min(result['response_evaluation']['completeness'], result['tool_evaluation']['tool_completeness']) < 80]
    case_evidence = [{
        'test_id': result['test_id'],
        'response_scores': {metric: result['response_evaluation'][metric] for metric in response_metrics},
        'tool_scores': {metric: result['tool_evaluation'][metric] for metric in ['tool_appropriateness', 'tool_completeness']},
        'response_feedback': result['response_evaluation']['feedback'],
        'tool_feedback': result['tool_evaluation']['feedback'],
    } for result in results]
    prompt = f'''You are preparing an honest evaluation summary for an energy advisor. Use only this scenario evidence: {json.dumps(case_evidence)}

Name specific test ids in every weakness. Give concrete project improvements that address the actual weak cases, such as tariff evidence, tool selection, quantified savings, or clearer assumptions. Do not give generic advice.'''
    try:
        narrative = run_judge(ReportJudgment, prompt)
        narrative['evaluation_method'] = 'llm_as_judge'
    except Exception as exc:
        named_cases = weak_cases or ['no scenario below the review threshold']
        narrative = {
            'strengths': [metric.replace('_', ' ') for metric, score in averages.items() if score >= 80] or ['No metric reached the strength threshold.'],
            'weaknesses': [f'{case}: review the saved response and tool trace.' for case in named_cases],
            'recommendations': [f'Review {case} and add the missing data evidence identified in its tool trace.' for case in named_cases],
            'evaluation_method': f'unavailable: {exc}',
        }
    return {
        'tests_completed': len(results),
        'overall_score': overall_score,
        'response_score': response_score,
        'tool_score': tool_score,
        'average_scores': averages,
        'strengths': narrative['strengths'],
        'weaknesses': narrative['weaknesses'],
        'needs_review': weak_cases,
        'recommendations': narrative['recommendations'],
        'evaluation_method': narrative['evaluation_method'],
    }

def display_evaluation_report(report):
    print('ECOHOME EVALUATION REPORT')
    print('=' * 28)
    print(f"Scenarios completed: {report['tests_completed']} | Overall score: {report['overall_score']}/100")
    print('\nMetric breakdown')
    for metric, score in report['average_scores'].items():
        print(f"  {metric.replace('_', ' ').title()}: {score}/100")
    print('\nStrengths')
    for item in report['strengths']:
        print(f'  • {item}')
    print('\nWeaknesses')
    for item in report['weaknesses']:
        print(f'  • {item}')
    print('\nRecommendations')
    for index, item in enumerate(report['recommendations'], start=1):
        print(f'  {index}. {item}')
    print(f"\nEvaluation method: {report['evaluation_method']}")


In [5]:
CONTEXT = 'Location: Berlin, Germany. Timezone: Europe/Berlin. Currency: EUR.'
test_results = []
for test_case in test_cases:
    print('Running:', test_case['id'])
    try:
        response = ecohome_agent.invoke(test_case['question'], CONTEXT)
        messages = response['messages']
        answer = final_text(messages)
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'final_response': answer,
            'tool_calls': get_tool_names(messages),
            'tool_trace': get_tool_trace(messages),
            'expected_tools': test_case['expected_tools'],
            'response_evaluation': evaluate_response(test_case['question'], answer, test_case['evaluation_criteria']),
            'tool_evaluation': evaluate_tool_usage(messages, test_case['expected_tools'], test_case['question']),
            'timestamp': datetime.now().isoformat(),
        }
    except Exception as exc:
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'final_response': '',
            'tool_calls': [],
            'tool_trace': [],
            'expected_tools': test_case['expected_tools'],
            'response_evaluation': {'accuracy': 0, 'relevance': 0, 'completeness': 0, 'usefulness': 0, 'feedback': [str(exc)]},
            'tool_evaluation': {'actual_tools': [], 'tool_appropriateness': 0, 'tool_completeness': 0, 'feedback': [str(exc)]},
            'timestamp': datetime.now().isoformat(),
            'error': str(exc),
        }
    test_results.append(result)

report = generate_evaluation_report(test_results)
Path('data/test_results.json').write_text(json.dumps(test_results, indent=2), encoding='utf-8')
Path('data/evaluation_report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
display_evaluation_report(report)
print(json.dumps(report, indent=2))


Running: ev_charging


Running: thermostat_peak


Running: dishwasher


Running: washing_machine


Running: pool_pump


...

Running: battery


Running: solar_history


Running: personalized_plan


Running: carbon_impact


ECOHOME EVALUATION REPORT
Scenarios completed: 12 | Overall score: 91.0/100

Metric breakdown
  Accuracy: 84.1/100
  Relevance: 92.5/100
  Completeness: 83.9/100
  Usefulness: 88.9/100
  Tool Appropriateness: 99.2/100
  Tool Completeness: 96.8/100

Strengths
  • Tool selection was consistently strong: most tests used appropriate combined planning, pricing, weather, history, and calculation routes, with tool-appropriateness generally scoring 100.
  • The recommendations were usually actionable and well tailored to Berlin timing, including solar windows, peak-price avoidance, EV departure constraints, comfort limits, and battery reserves.
  • The strongest responses were battery, solar_history, history_reduction, thermostat_peak, and carbon_impact: they were direct, practical, and generally transparent about uncertainty.
  • Several answers used appropriate caveats rather than guaranteeing savings or emissions reductions, especially in dishwasher, HVAC, solar-history, and carbon-impact c

## How to read the report

A high response score means the answer was clear, relevant, complete, and practical. A high tool score means the agent brought the right evidence into the answer. Any case listed under `needs_review` points to a response that should be improved before relying on it.